# Limpieza de `payment_events.csv`
Responsable: _(nombre)_

**Una fila del CSV = una entrega de una notificación de pago.** Los reenvíos comparten `provider_event_id` y no son cobros nuevos.

Pipeline: cargar → inspeccionar → tipar → validar → exportar a `data/clean/payment_events_clean.csv`.
El CSV original no se modifica.

In [ ]:
from pathlib import Path
import pandas as pd

# Sube desde el notebook hasta encontrar la raíz del repo (donde está el CSV)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "payment_events.csv").exists())
RAW = ROOT / "payment_events.csv"
CLEAN_DIR = ROOT / "data" / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

## 1. Carga

In [ ]:
raw = pd.read_csv(RAW, encoding="utf-8-sig")   # el CSV viene con BOM
print(raw.shape)
raw.head()

## 2. Inspección

In [ ]:
raw.info()
raw.isna().sum()

In [ ]:
print(raw["event_type"].value_counts())
print(raw["delivery_attempt"].value_counts())
raw["amount_eur"].describe()

## 3. Tipado

In [ ]:
df = raw.copy()
df["occurred_at"] = pd.to_datetime(df["occurred_at"], utc=True)
df["received_at"] = pd.to_datetime(df["received_at"], utc=True)
df.dtypes

## 4. Validaciones

In [ ]:
print("payment_event_id duplicados:", df["payment_event_id"].duplicated().sum())
print("importes <= 0:", (df["amount_eur"] <= 0).sum())
print("received_at < occurred_at:", (df["received_at"] < df["occurred_at"]).sum())
print("tipos inesperados:", (~df["event_type"].isin(["CAPTURED", "REFUNDED"])).sum())

## 5. Reenvíos
Se **conservan todas las filas**. Solo se marca cuáles son reenvíos; así nadie pierde información y,
al sumar dinero, se usa `provider_event_id` único.

In [ ]:
df["is_retry"] = df["delivery_attempt"] > 1
print("Reenvíos:", df["is_retry"].sum(), "de", len(df))
print("Notificaciones únicas (provider_event_id):", df["provider_event_id"].nunique())

## 6. Exportación

In [ ]:
out = df.copy()
for c in ["occurred_at", "received_at"]:
    out[c] = out[c].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
out.to_csv(CLEAN_DIR / "payment_events_clean.csv", index=False)
print("Guardado:", CLEAN_DIR / "payment_events_clean.csv", len(out), "filas")